In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

In [1]:
# If you see a pyarrow IpcReadOptions error, run:
#   pip install --upgrade pyarrow
# then restart the kernel before re-running.

import sys
import json
import os
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from openai import OpenAI

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv()

from tools import gcs_tools
from rag import retriever, dense
from tools.llm_tools import get_llm

from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
import ragas.metrics._context_precision as context_precision_metric
import ragas.metrics._context_recall as context_recall_metric
from ragas import evaluate
from ragas.llms import llm_factory

## Step 1 — Load evaluation dataset

In [2]:
dataset_path = Path.cwd().parent / "eval" / "dataset.json"
samples = json.loads(dataset_path.read_text(encoding="utf-8"))

counts = Counter(s["country_code"] for s in samples)
print("Questions per country:")
for country_code, count in sorted(counts.items()):
    print(f"  {country_code}: {count}")
print(f"Total: {len(samples)}")

Questions per country:
  CO: 12
  MX: 12
  PE: 12
Total: 36


## Step 2 — Metrics

Two retrieval quality metrics are used throughout this notebook:

**Context Precision**

$$\text{Context Precision} = \frac{\text{relevant chunks at } k}{\text{total chunks retrieved}}$$

Measures the signal-to-noise ratio in the retrieved context. A high score means most retrieved chunks are relevant to the question; a low score means the retriever returned a lot of noisy or off-topic chunks.

**Context Recall**

$$\text{Context Recall} = \frac{\text{reference claims covered by context}}{\text{total reference claims}}$$

Measures how much of the information required to answer the question was actually retrieved. A high score means the context contains most of the facts present in the ground-truth reference answer; a low score means key information was missed.

## Step 3 — Retrieval

In [3]:
retriever.initialize()


def retrieve_context(question: str, country_code: str, top_k: int = 5) -> list[str]:
    doc_ids = retriever.hybrid_retrieve(question, top_k=top_k, country_code=country_code)
    return [gcs_tools.get_document(doc_id) for doc_id in doc_ids]


def retrieve_context_dense_only(question: str, country_code: str, top_k: int = 5) -> list[str]:
    doc_ids = dense.search(question, top_k=top_k, country_code=country_code)
    return [gcs_tools.get_document(doc_id) for doc_id in doc_ids]

## Step 4 — Build RAGAS dataset

In [4]:
hybrid_ragas_samples = [
    SingleTurnSample(
        user_input=s["question"],
        retrieved_contexts=retrieve_context(s["question"], s["country_code"]),
        reference=s["reference"],
    )
    for s in samples
]
hybrid_dataset = EvaluationDataset(samples=hybrid_ragas_samples)
print(f"Built hybrid EvaluationDataset with {len(hybrid_dataset)} samples.")

Built hybrid EvaluationDataset with 36 samples.


## Step 5 — Evaluate hybrid retriever

In [5]:
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
ragas_llm = llm_factory(
    model=os.environ["RAGAS_LLM_MODEL"],
    client=openrouter_client,
)

context_precision_metric.llm = ragas_llm
context_recall_metric.llm = ragas_llm

hybrid_result = evaluate(
    dataset=hybrid_dataset,
    metrics=[context_precision_metric, context_recall_metric],
)

hybrid_df = hybrid_result.to_pandas()
hybrid_df["country_code"] = [s["country_code"] for s in samples]

print("=== Hybrid Retriever — Per-question scores ===")
print(hybrid_df[["country_code", "context_precision", "context_recall"]].to_string(index=False))

print("\n=== Hybrid Retriever — Mean scores per country ===")
print(
    hybrid_df.groupby("country_code")[["context_precision", "context_recall"]]
    .mean()
    .round(3)
)

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

=== Hybrid Retriever — Per-question scores ===
country_code  context_precision  context_recall
          CO           1.000000        1.000000
          CO           1.000000        1.000000
          CO           1.000000        1.000000
          CO           0.200000        1.000000
          CO           1.000000        1.000000
          CO           1.000000        1.000000
          CO           0.833333        1.000000
          CO           0.805556        1.000000
          CO           1.000000        1.000000
          CO           1.000000        1.000000
          CO           0.887500        1.000000
          CO           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.

## Step 6 — Evaluate dense-only retriever

In [6]:
dense_ragas_samples = [
    SingleTurnSample(
        user_input=s["question"],
        retrieved_contexts=retrieve_context_dense_only(s["question"], s["country_code"]),
        reference=s["reference"],
    )
    for s in samples
]
dense_dataset = EvaluationDataset(samples=dense_ragas_samples)
print(f"Built dense-only EvaluationDataset with {len(dense_dataset)} samples.")

dense_result = evaluate(
    dataset=dense_dataset,
    metrics=[context_precision_metric, context_recall_metric],
)

dense_df = dense_result.to_pandas()
dense_df["country_code"] = [s["country_code"] for s in samples]

print("=== Dense-Only Retriever — Per-question scores ===")
print(dense_df[["country_code", "context_precision", "context_recall"]].to_string(index=False))

print("\n=== Dense-Only Retriever — Mean scores per country ===")
print(
    dense_df.groupby("country_code")[["context_precision", "context_recall"]]
    .mean()
    .round(3)
)

Built dense-only EvaluationDataset with 36 samples.


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

=== Dense-Only Retriever — Per-question scores ===
country_code  context_precision  context_recall
          CO           1.000000        1.000000
          CO           1.000000        1.000000
          CO           1.000000        1.000000
          CO           0.200000        1.000000
          CO           0.887500        1.000000
          CO           1.000000        1.000000
          CO           0.833333        1.000000
          CO           0.950000        1.000000
          CO           1.000000        1.000000
          CO           0.833333        1.000000
          CO           0.950000        1.000000
          CO           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000        1.000000
          MX           1.000000      

## Step 7 — Comparison

In [7]:
hybrid_by_country = (
    hybrid_df.groupby("country_code")[["context_precision", "context_recall"]]
    .mean()
    .round(3)
    .add_prefix("hybrid_")
)
dense_by_country = (
    dense_df.groupby("country_code")[["context_precision", "context_recall"]]
    .mean()
    .round(3)
    .add_prefix("dense_")
)

comparison = pd.concat([dense_by_country, hybrid_by_country], axis=1)
comparison["precision_delta"] = (
    comparison["hybrid_context_precision"] - comparison["dense_context_precision"]
).round(3)
comparison["recall_delta"] = (
    comparison["hybrid_context_recall"] - comparison["dense_context_recall"]
).round(3)

print("=== Dense-Only vs Hybrid — by country ===")
print(comparison.to_string())

=== Dense-Only vs Hybrid — by country ===
              dense_context_precision  dense_context_recall  hybrid_context_precision  hybrid_context_recall  precision_delta  recall_delta
country_code                                                                                                                               
CO                              0.888                 1.000                     0.894                  1.000            0.006         0.000
MX                              0.991                 0.951                     1.000                  0.951            0.009         0.000
PE                              0.961                 0.979                     0.980                  0.938            0.019        -0.041


### Conclusion

Inspect `precision_delta` and `recall_delta` in the table above (hybrid minus dense-only):

- A **positive delta** means hybrid retrieval outperforms dense-only for that country.
- A **negative delta** means dense-only performed better.
- A delta near **0** means both strategies are equivalent for that country.

Overall, hybrid retrieval is expected to improve recall by surfacing chunks that dense embeddings rank poorly but sparse (BM25) keyword matching promotes — particularly for technical regulatory terms and document identifiers. Context precision may stay flat or decrease slightly if the sparse component introduces noisy results that dilute relevance.

Update this cell with actual observed values once the evaluation has run.